# 00 はじめに — 基礎知識と用語から始めよう（時系列予測と深層学習）

> 🔒 すべて**合成データ**です。⏱ 所要時間: 読む 30分 ＋ 実行 数秒。**難しければ、「図の見方」と「用語集」だけ読んで、次のNotebookへ進んでください。**

このNotebookは、**予備知識ゼロ**から始めます。数式は、ほとんど使いません。分からない言葉が出てきたら、いつでもここに戻ってきてください。

## 1. このプロジェクトは何をするの？（ひとことで）
**「明日の太陽光発電量は、何kWhか」を当てる**練習です。天気予報（日射量の予報）と、過去の発電量を使います。
そして、**「流行りの深層学習（ディープラーニング）は、簡単な方法より本当に当たるのか？」** を、自分の手で確かめます。

たとえ話: 明日の傘が要るかを当てるとき、
- 「昨日と同じだろう」と考える（→ **持続予報**。一番かんたん）
- 「天気予報が雨なら持っていく」と考える（→ **予報を使う**）
- 「この季節・この雲の出方・昨日の天気…を総合して」考える（→ **機械学習・深層学習**）
どこまで手の込んだ方法にすると、どれだけ当たるようになるのか。それを数字で比べます。

## 2. 基礎知識（これだけ分かれば読めます）

### 2-1. 時系列とは
**時間の順に並んだ数字**のこと。毎日の発電量、毎時の気温、株価など。大事な性質が2つあります。
- **隣の日は似ている**（昨日が晴れなら、今日も晴れやすい）。
- **季節の波がある**（夏は多く、冬は少ない）。
このため、データを**ランダムに混ぜて**分けてはいけません（未来の情報が混ざってしまう）。必ず**時間の順**に、過去で学び、未来で試します。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]

rng = np.random.default_rng(0)
days = np.arange(365 * 2)
season = 8 + 6 * np.sin(2 * np.pi * (days - 80) / 365)          # 季節の波
weather = rng.normal(0, 3, len(days))                            # 日ごとの天気のばらつき
gen = np.clip(season + weather, 0, None)
plt.figure(figsize=(11, 3.2))
plt.plot(days, gen, lw=0.7, label="毎日の発電量（例）")
plt.plot(days, season, lw=2, label="季節の波（ゆっくりした変化）")
plt.xlabel("日"); plt.ylabel("発電量 kWh/日"); plt.legend(); plt.grid(alpha=0.3); plt.show()

**図の見方**: 細い線が毎日の発電量。太い線が「季節の波」です。毎日の値は、**季節の波 ＋ その日の天気によるぶれ**でできています。予測とは、このうち**ぶれの部分をどこまで当てられるか**の勝負です。

### 2-2. 「誤差」を測る — MAE
予測がどれだけ外れたかを、1つの数字にするのが**誤差の指標**です。このプロジェクトでは **MAE（平均絶対誤差）** を使います。
- 各日について「予測 − 実際」を計算し、**プラスマイナスを無視して**（絶対値）、**平均**する。
- 単位は kWh/日（発電量と同じ）。**小さいほど良い**。MAE=1 なら「平均して1kWhずれる」。

In [ ]:
actual = np.array([10.0, 12.0, 6.0, 9.0])    # 実際の発電量（例）
pred   = np.array([11.0, 10.0, 7.0, 9.0])    # 予測（例）
err = pred - actual
print("各日のずれ:", err)
print("絶対値:", np.abs(err))
print("MAE =", np.abs(err).mean(), "kWh/日")

**出力の見方**: ずれは +1, −2, +1, 0。絶対値にすると 1, 2, 1, 0 で、平均して **MAE = 1.0**。プラスとマイナスで打ち消し合わないように、絶対値を使うのがコツです。

### 2-3. 基準（ベースライン）とスキル
新しい方法が良いかは、**一番かんたんな方法との比較**で決めます。
- **持続予報**: 「昨日の値」をそのまま予測にする。
- **スキル** = `1 − (自分のMAE) ÷ (持続予報のMAE)`。0 なら持続予報と同じ、0.5 なら誤差が半分。
**複雑な手法が、かんたんな基準に負ける**ことは、珍しくありません。基準を先に作るのは、そのためです。

### 2-4. 直線で予測する（線形回帰）
入力 x（例: 予報の日射量）と答え y（発電量）の関係を、**直線 y = a + b·x** で近づけます。a, b をデータから決めること＝**学習（当てはめ）**。

In [ ]:
rng = np.random.default_rng(1)
x = rng.uniform(2, 28, 60)                         # 日射量の予報（例）
y = 0.85 * x + rng.normal(0, 1.5, 60)              # 発電量（例）: だいたい0.85倍＋ばらつき
b, a = np.polyfit(x, y, 1)                          # 直線を当てはめる
xs = np.linspace(2, 28, 50)
plt.figure(figsize=(5.5, 4)); plt.scatter(x, y, s=12, label="データ"); plt.plot(xs, a + b * xs, "r", label=f"直線 y={a:.2f}+{b:.2f}x")
plt.xlabel("日射量の予報"); plt.ylabel("発電量"); plt.legend(); plt.grid(alpha=0.3); plt.show()

**図の見方**: 点がデータ、赤線が学習した直線です。点は直線の周りに散らばります。**この散らばり（直線で説明できない部分）が、予測の限界**です。直線の傾き b ≈ 0.85 は、「日射量1あたり0.85kWh発電する」という物理の関係に近い値です。
このプロジェクトの出発点（基準）が、この**予報の直線回帰**です。

### 2-5. 機械学習とは
直線のように人が形を決めるのではなく、**データから関数の形そのものを学ぶ**方法です。
- **木（決定木・ランダムフォレスト・勾配ブースティング）**: 「日射量が10より大きいか？」「夏か？」…と**質問を重ねて**答えを出す。表のデータが得意。
- **ニューラルネットワーク（深層学習）**: 小さな計算部品（ニューロン）をたくさん重ねた関数。画像・音声・文章が得意と言われる。

### 2-6. ニューラルネットワークのしくみ（超ざっくり）
1つのニューロンは「入力を重み付きで足し合わせ、**ReLU**（マイナスなら0にする）に通す」だけです。これを**重ねる**と、折れ線を組み合わせた**なめらかな曲線**が作れます。

In [ ]:
relu = lambda z: np.maximum(z, 0)
x = np.linspace(-3, 3, 200)
target = x ** 2                                              # 当てたい曲線（例）
approx = relu(x) + 2 * relu(x - 1) + 2 * relu(x - 2) + relu(-x) + 2 * relu(-x - 1) + 2 * relu(-x - 2)   # ReLUを6つ足しただけ
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(x, relu(x)); ax[0].set_title("ReLU（部品）：マイナスを0にするだけ"); ax[0].grid(alpha=0.3)
ax[1].plot(x, target, label="当てたい曲線 y=x²"); ax[1].plot(x, approx, "--", label="ReLU 6個の足し算"); ax[1].legend(); ax[1].grid(alpha=0.3)
ax[1].set_title("部品を重ねると曲線に近づく"); plt.tight_layout(); plt.show()

**図の見方**: 左は部品のReLU（折れ曲がった1本の線）。右は、たった6個を（重みを付けて）足しただけで、曲線 y=x² にかなり近づく様子です（折れ線ですが、なめらかな曲線にそっくりです）。**部品を増やすほど、複雑な形を表せます**。ただし、増やしすぎると次の「過学習」が起きます。

**学習**とは、重み（足し合わせの係数）を、**誤差が小さくなる方向に少しずつ**直していくことです。1周分のデータを使い切ることを**エポック**と呼び、何周も繰り返します。

### 2-7. 過学習 — いちばん大事な落とし穴
学習に使ったデータには**ぴったり**合うのに、**新しいデータには外れる**ことを過学習といいます。試験で、過去問の答えを丸暗記して、問題が少し変わると解けなくなるのと同じです。

In [ ]:
rng = np.random.default_rng(3)
f = lambda x: np.sin(x)
xt = np.sort(rng.uniform(0, 6, 12)); yt = f(xt) + rng.normal(0, 0.3, 12)      # 学習データ（少ない・ノイズあり）
xv = np.sort(rng.uniform(0, 6, 200)); yv = f(xv) + rng.normal(0, 0.3, 200)     # 新しいデータ
xs = np.linspace(0, 6, 200)
fig, ax = plt.subplots(1, 3, figsize=(14, 3.4), sharey=True)
for a_, deg in zip(ax, [1, 3, 9]):
    p = np.polyfit(xt, yt, deg)
    mae_t = np.abs(np.polyval(p, xt) - yt).mean(); mae_v = np.abs(np.polyval(p, xv) - yv).mean()
    a_.scatter(xt, yt, s=18, c="k"); a_.plot(xs, np.polyval(p, xs), "r"); a_.set_ylim(-2.5, 2.5)
    a_.set_title(f"曲がり具合={deg}\n学習の誤差 {mae_t:.2f} ／ 新データの誤差 {mae_v:.2f}"); a_.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**図の見方**（黒点が学習データ、赤線が学習した曲線）:
- **曲がり具合=1（直線）**: 単純すぎて、どちらの誤差も大きい（**当てはまりが足りない**）。
- **曲がり具合=3**: ちょうど良く、新データの誤差も小さい。
- **曲がり具合=9**: 学習データの点をほぼ全部通る（学習の誤差が最小）が、点と点の間で**暴れて**、新データの誤差は悪化。**これが過学習**です。
**学習の誤差が小さい＝良いモデル、ではありません。** 必ず、**学習に使っていないデータ**で確かめます。そのために、データを**学習・検証・テスト**に分けます。

### 2-8. データの分け方（時間の順に）
| 名前 | 役割 | このプロジェクトでの例 |
|---|---|---|
| **学習（train）** | 重みを学ぶ | 過去10年 |
| **検証（val）** | 設定（ネットの大きさなど）を選ぶ、学習をやめる時期を決める | 次の2年 |
| **テスト（test）** | 最後に**1回だけ**、本当の実力を測る | 最後の2年 |
テストの結果を見て設定を変えてはいけません（見た時点で、もう「未使用」ではなくなるため）。

## 3. 用語集（ここに戻ってきてください）

| 用語 | やさしい説明 |
|---|---|
| **発電量 (kWh)** | 1日に発電した電気の量。 |
| **日射量 (GSR)** | 空から届く太陽のエネルギーの量。多いほど発電量が多い。 |
| **予報とリード L** | 「L日前に出した予報」。L=1は前日の夜に出した予報。**先の日ほど外れやすい**。 |
| **持続予報** | 昨日の値をそのまま使う、最も簡単な基準。 |
| **平年値** | 同じ日付の、過去の平均。 |
| **MAE** | 誤差の平均（絶対値）。小さいほど良い。 |
| **標準誤差** | 平均の「あやふやさ」。**2つの手法の差が、標準誤差の2倍より小さいときは、差があるとは言えない**。 |
| **特徴量** | 予測に使う入力の数字（予報、昨日の発電量、季節など）。 |
| **ラグ特徴量** | 「何日前の値」という特徴量。 |
| **リーク** | 予測する時点では分からない情報（未来・答え）を、入力にうっかり混ぜること。見かけの成績だけが良くなる、最も危険なミス。 |
| **隠れた状態** | 直接は見えないが、過去の並びから推測できる要因。ここでは、パネルの「よごれ」（雨で洗い流される）。 |
| **木・ランダムフォレスト・HGB** | 質問を重ねて答えを出す手法。HGBは勾配ブースティング。 |
| **MLP** | 全結合のニューラルネット（一番基本的な形）。 |
| **GRU** | 時系列を**1日ずつ順に読み、記憶を更新していく**ニューラルネット。積み重なる状態に向く。 |
| **1D-CNN** | 時系列の中の短いパターンを拾うニューラルネット。 |
| **エポック / 早期終了** | 学習データ1周 / 検証の誤差が最小になった時点で学習をやめること（過学習を防ぐ）。 |
| **初期値のぶれ・アンサンブル** | ニューラルネットは、最初の重みの乱数で結果がぶれる。複数の初期値の予測を**平均**するとぶれが減る。 |
| **分位点・予測区間** | 「実測の90%がこれ以下」となる値／「80%の確率でこの範囲」という幅。 |
| **カバー率** | 予測区間に実測が入った割合。80%区間なら0.8に近いほど「正直」。 |
| **pinball loss** | 分位点を学習するための損失。 |
| **合成データ** | プログラムで作った練習用データ。**答え（隠れた状態など）が分かる**ので、手法を正しく試せる。 |

## 4. このプロジェクトの地図（読む順）

| Notebook | 問い | 一言 |
|---|---|---|
| 00 データと基準 | 簡単な方法で、どこまで当たる？ | 予報が一番効く |
| 01 古典的な機械学習 | 特徴量を足すと良くなる？ | 少し良くなるが、予報の誤差に埋もれる |
| 02 MLP | ニューラルネットは木に勝つ？ | 勝てない（ぶれが大きい） |
| 03 系列モデル | 生の時系列から、隠れた状態を読める？ | 予報が良ければ読める |
| 04 データ量とノイズ | データが少ないとどうなる？ | 木が先に過学習する |
| 05 確率予測 | 「範囲」で答えられる？ | ニューラルネットが正直な区間を出す |

## 5. 結果を読むときの心得（5つ）
1. **基準と比べる。** 単独の数字（MAE=0.8）に意味はない。
2. **標準誤差と並べる。** 差が小さければ「同じ」と考える。
3. **学習に使っていないデータの成績を見る。**
4. **1回の結果を信じない**（複数の世界・複数の初期値の平均）。
5. **合成データの結論を、そのまま現実に持ち込まない。**

## 6. よくある疑問
- **Q. 深層学習は、一番当たるのでは？** A. 画像や文章では強いですが、このような表形式のデータでは、木や直線と同じくらいのことが多い。ここで、数字で確かめます。
- **Q. なぜ合成データ？** A. 「隠れた状態」の正解を知っているので、モデルがそれを**本当に読めているか**を確かめられるからです。
- **Q. 数式が出てきたら？** A. 各Notebookの「出力の見方」の文章だけ追えば大丈夫です。

次は **00_data_and_baselines** へ進みましょう。